# Convert and downsample whole-slide images

Runs `run_conversion.py` from this notebook. Use the **Python (histology-pipeline)** kernel
(created from `environment.yml` at the repository root).

1. Edit the settings cell.
2. Run all cells. Each source image is read once at the finest requested resolution,
   then every output folder is written from it. Images that already exist in an output
   folder are skipped, so a stopped run can simply be restarted.

In [ ]:
import sys
from pathlib import Path

# Make the conversion scripts importable regardless of where Jupyter was started.
HERE = Path.cwd()
SCRIPT_DIR = HERE if (HERE / 'run_conversion.py').exists() else HERE / '01_conversion_downsampling'
sys.path.insert(0, str(SCRIPT_DIR))

from run_conversion import run_conversion
print('Python:', sys.executable)

## Settings

- `file_format`: `'vsi'`, `'czi'`, or `'other'` (Pramana `.dcm`, `.ndpi`, `.svs`, `.mrxs`, `.tif`, iSyntax, ...).
- `folder_names`, `pixel_resolutions`, `save_ome` correspond by position.
- `pixel_resolutions` are in micrometres/pixel; `0` keeps native resolution.
- `save_ome`: `0` = plain TIFF, `1` = OME-TIFF.
- `output_path = None` writes the subfolders inside the input folder.

In [ ]:
input_path = r'\\10.17.182.53\kiemen-lab-data\Marta Pereira\Multi scanner project\Scanned images\Scanner types\Pramana_40x'
output_path = None
file_format = 'other'
folder_names = ['2x', '40x']
pixel_resolutions = [5, 0.25]
save_ome = [0, 1]
scanner_name = 'Pramana'
scanner_manifest = None           # Optional CSV with image,scanner columns for mixed batches
load_native_resolution = 1        # 1 = native decoding; 0 = allow the source pyramid (faster)

## Check the input

In [ ]:
source = Path(input_path)
print('Input exists:', source.exists())
extensions = {'.vsi', '.czi', '.ndpi', '.ndp', '.svs', '.scn', '.mrxs', '.dcm', '.qptiff',
              '.tif', '.tiff', '.isyntax', '.i2syntax'}
images = sorted(f.name for f in source.iterdir() if f.is_file() and f.suffix.lower() in extensions) \
    if source.is_dir() else [source.name]
print(f'{len(images)} candidate image(s):')
for name in images[:10]:
    print('  ', name)
if len(images) > 10:
    print(f'   ... and {len(images) - 10} more')

## Run the conversion

In [ ]:
run_conversion(input_path, output_path, file_format, folder_names, pixel_resolutions,
               save_ome, load_native_resolution, scanner_name, scanner_manifest)

## Results

In [ ]:
import csv

out_root = Path(output_path) if output_path else (source if source.is_dir() else source.parent)
for folder in folder_names:
    files = sorted((out_root / folder).glob('*.tif*'))
    print(f'{folder}: {len(files)} file(s)')
    for f in files:
        print(f'   {f.name}  ({f.stat().st_size / 1e6:,.1f} MB)')

# Most recent per-image timing log written by this run
logs = sorted((out_root / 'timings').glob('*.csv'), key=lambda p: p.stat().st_mtime)
if logs:
    print('\nTiming log:', logs[-1])
    with open(logs[-1], newline='', encoding='utf-8') as stream:
        for row in csv.DictReader(stream):
            print('  ', {k: v for k, v in row.items() if v})